# Traditional-Media-Only Models — Predicting Daily Change in Polymarket Trump Probability

Four regression models trained exclusively on newspaper / MediaCloud features:
1. **Ridge Regression** — regularised linear baseline; alpha tuned by CV  
   *(Plain OLS is nearly singular in the first CV fold where n_train ≈ 24 ≈ 23 features — Ridge prevents unstable solutions.)*
2. **Random Forest** — tree ensemble; tuned `max_depth`, `min_samples_leaf`
3. **SVM** — kernel regression; tuned `kernel`, `C`, `epsilon`
4. **XGBoost** — gradient boosting; tuned `max_depth`, `learning_rate`, `n_estimators`

**Target:** `Δ polymarket_trump_prob = prob(t) − prob(t−1)` — daily change in Trump win probability  
**Features (23):** newspaper sentiment (TextBlob), FinBERT financial sentiment, and topic-share columns.

> **Note on contemporaneous features.** All 23 newspaper features are same-day aggregates: articles published on day *t* are used to predict the polymarket change on day *t*. They are not leaked in the pipeline sense (scalers fitted on train folds only; test set never touched during model selection), but the model relies on information that is only fully available at the end of day *t*.

**Dataset coverage:** newspaper data is available for ~111 days (some days have no articles); rows with any NaN are dropped.

**Hyperparameter tuning:** grid search using the same 3 walk-forward CV folds.  
**Splits:** walk-forward expanding-window CV, 3 folds, gap = 1 day, held-out test = last 14 days  
**Metrics:** MAE, RMSE, Directional Accuracy, R²

## 1. Setup

In [ ]:
import sys, os, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, "../../..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error
try:
    from xgboost import XGBRegressor
except ImportError:
    raise ImportError("XGBoost not found. Install with: pip install xgboost")

from Functions.data_splits import (
    get_cv_folds, get_test_split, print_fold_summary, validate_no_leakage,
)
from Functions.evaluation_metrics import (
    directional_accuracy, compute_metrics, cv_evaluate, final_eval, tune_hyperparams,
)
from house_style import (
    apply_style, BG_DARK, BG_PANEL, TEXT_PRIMARY, TEXT_MUTED,
    SPINE_COLOR, GRID_COLOR, DEMOCRAT, REPUBLICAN, NEUTRAL, PALETTE,
)
apply_style()

RANDOM_STATE = 42
TEST_DAYS    = 14
N_SPLITS     = 3
GAP          = 1

DATA_PATH = "../../../Data/3_Gold/basetable.csv"

NEWS_COLS = [
    # ── TextBlob sentiment ───────────────────────────────────────────────────────────────
    "news_title_trump_count",
    "news_attention_asymmetry",
    "news_trump_sentiment_avg",
    "news_trump_sent_std",
    "news_trump_neg_ratio",
    "news_trump_strong_pos_ratio",
    "news_trump_strong_neg_ratio",
    "news_harris_sentiment_avg",
    "news_harris_sent_std",
    "news_harris_neg_ratio",
    "news_harris_strong_pos_ratio",
    "news_harris_strong_neg_ratio",
    # ── FinBERT financial sentiment ───────────────────────────────────────────────────
    "news_trump_finbert_avg",
    "news_trump_finbert_std",
    "news_trump_finbert_pos_ratio",
    "news_trump_finbert_neg_ratio",
    "news_harris_finbert_avg",
    "news_harris_finbert_std",
    "news_harris_finbert_pos_ratio",
    "news_harris_finbert_neg_ratio",
    "news_finbert_gap",
    # ── Topic shares ───────────────────────────────────────────────────────────────────────
    "topic_economy_share",
    "topic_immigration_share",
]

MODEL_COLORS = {
    "Naive (zero)"    : NEUTRAL,
    "Ridge Regression": PALETTE[0],
    "Random Forest"   : PALETTE[1],
    "SVM"             : PALETTE[4],
    "XGBoost"         : PALETTE[2],
}

print("Imports OK")

## 2. Load Data & Compute Target

We load `basetable.csv` and keep only the 23 newspaper columns.

**Target derivation:**  
`target(t) = polymarket_trump_prob(t) − polymarket_trump_prob(t−1)`

Computed via explicit shift; the first row and any rows with NaN newspaper features are removed by `dropna()`.

In [ ]:
df_raw = pd.read_csv(DATA_PATH, parse_dates=["date"])
df_raw = df_raw.sort_values("date").reset_index(drop=True)

missing = [c for c in NEWS_COLS if c not in df_raw.columns]
assert not missing, f"Missing newspaper columns: {missing} — check basetable.csv"

df = df_raw[["date", "polymarket_trump_prob"] + NEWS_COLS].copy()
df["target"] = df["polymarket_trump_prob"] - df["polymarket_trump_prob"].shift(1)
df = df.dropna().reset_index(drop=True)  # removes first row (NaN target) + days with no newspaper data

print(f"Rows          : {len(df)}")
print(f"Date range    : {df['date'].min().date()} -> {df['date'].max().date()}")
print(f"Features ({len(NEWS_COLS)}): {len(NEWS_COLS)} newspaper columns")
print(f"\nTarget (daily delta prob) stats:")
print(df["target"].describe().round(5))

## 3. Train/Val/Test Split

`get_test_split` carves off the **last 14 rows** as a completely held-out test set.

In [ ]:
tv_idx, test_idx = get_test_split(df, test_days=TEST_DAYS)

df_tv   = df.iloc[tv_idx].reset_index(drop=True)
df_test = df.iloc[test_idx].reset_index(drop=True)

X_tv   = df_tv[NEWS_COLS].values
y_tv   = df_tv["target"].values
X_test = df_test[NEWS_COLS].values
y_test = df_test["target"].values

print(f"Train/val : {len(df_tv):>4} rows  ({df_tv['date'].min().date()} -> {df_tv['date'].max().date()})")
print(f"Test      : {len(df_test):>4} rows  ({df_test['date'].min().date()} -> {df_test['date'].max().date()})")

## 4. CV Folds

**3 expanding-window folds** on the TV set with a 1-day gap.

In [ ]:
folds = get_cv_folds(df_tv, n_splits=N_SPLITS, gap=GAP, test_days=None)
print_fold_summary(df_tv, folds)

print("\nLeakage validation:")
for i, (tr, va) in enumerate(folds, 1):
    validate_no_leakage(tr, va, df_tv, gap=GAP)
    print(f"  Fold {i}: OK")

## 5. Helper Functions

All helpers are imported from `Functions.evaluation_metrics`. Scalers in `cv_evaluate` and `final_eval` are fitted exclusively on training data.

## 6. Model 1 — Ridge Regression

Ridge adds an L2 penalty: `β̂ = argmin ‖y − Xβ‖² + α‖β‖²`. This is critical here because the first CV fold has only ~24 training rows for 23 features — plain OLS would have effectively zero degrees of freedom and highly unstable coefficients. Ridge keeps the solution numerically stable regardless of near-singularity.

**Tuned hyperparameter:** `alpha` — regularisation strength.  
**Scaling applied** — needed for the L2 penalty to treat all features equally.

In [ ]:
def make_ridge(alpha):
    return Ridge(alpha=alpha)

ridge_param_grid = {"alpha": [0.01, 0.1, 1.0, 10.0, 100.0, 500.0, 1000.0]}

print("=== Ridge Regression — Hyperparameter Tuning ===")
ridge_best, ridge_tune_df = tune_hyperparams(
    make_ridge, ridge_param_grid, folds, X_tv, y_tv, scale=True
)
print(f"  Best params : {ridge_best}")
print(f"\n  All configurations (sorted by CV MAE):")
print(ridge_tune_df.to_string(index=False))

ridge_factory = lambda: make_ridge(**ridge_best)
print("\n=== Ridge Regression — CV (best params) ===")
ridge_cv = cv_evaluate(ridge_factory, folds, X_tv, y_tv, scale=True)
ridge_cv.round(4)

In [ ]:
ridge_model, ridge_pred, ridge_test = final_eval(
    ridge_factory, X_tv, y_tv, X_test, y_test, scale=True
)
print(f"Ridge Regression — Test set  (alpha={ridge_best['alpha']}):")
for k, v in ridge_test.items():
    print(f"  {k}: {v:.4f}")

## 7. Model 2 — Random Forest Regressor

Random Forest is robust to small training sets because each tree uses a bootstrap sample and a random subset of features. Decision-tree splits are scale-invariant and immune to near-singular feature matrices.

**Fixed:** `n_estimators=200`, `random_state=42`.  
**Tuned:** `max_depth`, `min_samples_leaf`.  
**No scaling required.**

In [ ]:
def make_rf(max_depth, min_samples_leaf):
    return RandomForestRegressor(
        n_estimators=200,
        max_depth=int(max_depth),
        min_samples_leaf=int(min_samples_leaf),
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )

rf_param_grid = {
    "max_depth"       : [2, 3, 4, 5],
    "min_samples_leaf": [1, 2, 3, 5],
}

print("=== Random Forest — Hyperparameter Tuning ===")
rf_best, rf_tune_df = tune_hyperparams(
    make_rf, rf_param_grid, folds, X_tv, y_tv, scale=False
)
print(f"  Best params : {rf_best}")
print(f"\n  Top-5 configurations (sorted by CV MAE):")
print(rf_tune_df.head(5).to_string(index=False))

rf_factory = lambda: make_rf(**rf_best)
print("\n=== Random Forest — CV (best params) ===")
rf_cv = cv_evaluate(rf_factory, folds, X_tv, y_tv, scale=False)
rf_cv.round(4)

In [ ]:
rf_model, rf_pred, rf_test = final_eval(
    rf_factory, X_tv, y_tv, X_test, y_test, scale=False
)
print("Random Forest — Test set:")
for k, v in rf_test.items():
    print(f"  {k}: {v:.4f}")

fi = pd.Series(rf_model.feature_importances_, index=NEWS_COLS).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
fig.patch.set_facecolor(BG_DARK)
ax.set_facecolor(BG_PANEL)
bars = ax.barh(fi.index, fi.values, color=MODEL_COLORS["Random Forest"], alpha=0.85, height=0.6)
for bar, v in zip(bars, fi.values):
    ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
            f"{v:.3f}", va="center", ha="left", fontsize=8, color=TEXT_MUTED)
ax.set_xlabel("Mean Decrease in Impurity", color=TEXT_PRIMARY)
ax.set_title(f"Random Forest — Feature Importances  |  best: {rf_best}",
             color=TEXT_PRIMARY, fontweight="bold", fontsize=9)
for spine in ax.spines.values(): spine.set_edgecolor(SPINE_COLOR)
ax.tick_params(colors=TEXT_MUTED)
ax.set_yticklabels(fi.index, color=TEXT_PRIMARY, fontsize=8)
ax.grid(axis="x", color=GRID_COLOR, linewidth=0.7)
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()

## 8. Model 3 — Support Vector Regression (tuned kernel)

SVR is well-suited to medium-dimensional data (23 features) — the dual formulation depends on the number of support vectors rather than features, so it scales better than explicit feature expansion.

**Fixed:** `gamma='scale'`.  
**Tuned:** `kernel`, `C`, `epsilon`.  
**Scaling required.**

In [ ]:
def make_svr(kernel, C, epsilon):
    return SVR(kernel=kernel, C=C, epsilon=epsilon, gamma="scale")

svr_param_grid = {
    "kernel" : ["linear", "rbf", "poly", "sigmoid"],
    "C"      : [0.1, 1.0, 10.0, 100.0],
    "epsilon": [0.0001, 0.001, 0.01, 0.05],
}

print("=== SVM — Hyperparameter Tuning ===")
svr_best, svr_tune_df = tune_hyperparams(
    make_svr, svr_param_grid, folds, X_tv, y_tv, scale=True
)
print(f"  Best params : {svr_best}")
print(f"\n  Top-10 configurations (sorted by CV MAE):")
print(svr_tune_df.head(10).to_string(index=False))

svr_factory = lambda: make_svr(**svr_best)
print("\n=== SVM — CV (best params) ===")
svm_cv = cv_evaluate(svr_factory, folds, X_tv, y_tv, scale=True)
svm_cv.round(4)

In [ ]:
svm_model, svm_pred, svm_test = final_eval(
    svr_factory, X_tv, y_tv, X_test, y_test, scale=True
)
print(f"SVM — Test set  (kernel={svr_best['kernel']}, C={svr_best['C']}, ε={svr_best['epsilon']}):")
for k, v in svm_test.items():
    print(f"  {k}: {v:.4f}")

## 9. Model 4 — XGBoost Regressor

XGBoost handles correlated newspaper sentiment/FinBERT features naturally via the boosting objective: redundant features simply contribute less gain at each split.

**Fixed:** `subsample=0.8`, `colsample_bytree=0.8`, `objective='reg:squarederror'`, `random_state=42`.  
**Tuned:** `max_depth`, `learning_rate`, `n_estimators`.  
**No scaling required.**

In [ ]:
def make_xgb(max_depth, learning_rate, n_estimators):
    return XGBRegressor(
        max_depth=int(max_depth),
        learning_rate=learning_rate,
        n_estimators=int(n_estimators),
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=RANDOM_STATE,
        verbosity=0,
    )

xgb_param_grid = {
    "max_depth"    : [2, 3, 4],
    "learning_rate": [0.01, 0.05, 0.1],
    "n_estimators" : [100, 200],
}

print("=== XGBoost — Hyperparameter Tuning ===")
xgb_best, xgb_tune_df = tune_hyperparams(
    make_xgb, xgb_param_grid, folds, X_tv, y_tv, scale=False
)
print(f"  Best params : {xgb_best}")
print(f"\n  Top-5 configurations (sorted by CV MAE):")
print(xgb_tune_df.head(5).to_string(index=False))

xgb_factory = lambda: make_xgb(**xgb_best)
print("\n=== XGBoost — CV (best params) ===")
xgb_cv = cv_evaluate(xgb_factory, folds, X_tv, y_tv, scale=False)
xgb_cv.round(4)

In [ ]:
xgb_model, xgb_pred, xgb_test = final_eval(
    xgb_factory, X_tv, y_tv, X_test, y_test, scale=False
)
print("XGBoost — Test set:")
for k, v in xgb_test.items():
    print(f"  {k}: {v:.4f}")

fi_xgb = pd.Series(xgb_model.feature_importances_, index=NEWS_COLS).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
fig.patch.set_facecolor(BG_DARK)
ax.set_facecolor(BG_PANEL)
bars = ax.barh(fi_xgb.index, fi_xgb.values, color=MODEL_COLORS["XGBoost"], alpha=0.85, height=0.6)
for bar, v in zip(bars, fi_xgb.values):
    ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
            f"{v:.3f}", va="center", ha="left", fontsize=8, color=TEXT_MUTED)
ax.set_xlabel("Feature Importance (gain)", color=TEXT_PRIMARY)
ax.set_title(f"XGBoost — Feature Importances (gain)  |  best: {xgb_best}",
             color=TEXT_PRIMARY, fontweight="bold", fontsize=9)
for spine in ax.spines.values(): spine.set_edgecolor(SPINE_COLOR)
ax.tick_params(colors=TEXT_MUTED)
ax.set_yticklabels(fi_xgb.index, color=TEXT_PRIMARY, fontsize=8)
ax.grid(axis="x", color=GRID_COLOR, linewidth=0.7)
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()

## 10. Naive Baseline — Always Predict Zero

The naive model always predicts `Δ prob = 0`. Any model with R² < 0 is strictly worse than this no-information baseline.

In [ ]:
print("=== Naive (zero) — CV ===")
naive_records = []
for i, (train_idx, val_idx) in enumerate(folds, 1):
    y_val  = y_tv[val_idx]
    y_zero = np.zeros_like(y_val)
    m = {"Fold": i, **compute_metrics(y_val, y_zero)}
    naive_records.append(m)
    print(f"  Fold {i}: MAE={m['MAE']:.4f}  RMSE={m['RMSE']:.4f}  "
          f"DA={m['Dir. Accuracy']:.3f}  R2={m['R2']:.4f}")

naive_agg  = pd.DataFrame(naive_records).set_index("Fold")
naive_mean = naive_agg.mean().rename("Mean")
naive_std  = naive_agg.std().rename("Std")
naive_cv   = pd.concat([naive_agg, naive_mean.to_frame().T, naive_std.to_frame().T])
print(f"  -- Mean --  MAE={naive_mean['MAE']:.4f}  RMSE={naive_mean['RMSE']:.4f}  "
      f"DA={naive_mean['Dir. Accuracy']:.3f}  R2={naive_mean['R2']:.4f}")

naive_pred = np.zeros(len(y_test))
naive_test = compute_metrics(y_test, naive_pred)
print("\nNaive (zero) — Test set:")
for k, v in naive_test.items():
    print(f"  {k}: {v:.4f}")

naive_cv.round(4)

## 11. Model Comparison

CV performance (mean over 3 folds) and test-set performance for all five models.

In [ ]:
cv_summary = pd.DataFrame({
    "Naive (zero)"    : naive_cv.loc["Mean"],
    "Ridge Regression": ridge_cv.loc["Mean"],
    "Random Forest"   : rf_cv.loc["Mean"],
    "SVM"             : svm_cv.loc["Mean"],
    "XGBoost"         : xgb_cv.loc["Mean"],
}).T.round(4)

best_params_col = {
    "Naive (zero)"    : "— (always 0)",
    "Ridge Regression": f"alpha={ridge_best['alpha']}",
    "Random Forest"   : f"d={rf_best['max_depth']}, leaf={rf_best['min_samples_leaf']}",
    "SVM"             : f"k={svr_best['kernel']}, C={svr_best['C']}, ε={svr_best['epsilon']}",
    "XGBoost"         : f"d={xgb_best['max_depth']}, lr={xgb_best['learning_rate']}, n={int(xgb_best['n_estimators'])}",
}
cv_summary.insert(0, "Best params", pd.Series(best_params_col))

print("CV performance (mean across 3 walk-forward folds):")
display(cv_summary)

In [ ]:
test_summary = pd.DataFrame({
    "Naive (zero)"    : naive_test,
    "Ridge Regression": ridge_test,
    "Random Forest"   : rf_test,
    "SVM"             : svm_test,
    "XGBoost"         : xgb_test,
}).T.round(4)

print("Test set performance:")
display(test_summary)

In [ ]:
test_dates = df_test["date"].values
preds_list = [
    ("Naive (zero)",    naive_pred),
    ("Ridge Regression", ridge_pred),
    ("Random Forest",   rf_pred),
    ("SVM",             svm_pred),
    ("XGBoost",         xgb_pred),
]

fig, axes = plt.subplots(5, 1, figsize=(13, 12), sharex=True)
fig.patch.set_facecolor(BG_DARK)

for ax, (label, pred) in zip(axes, preds_list):
    ax.set_facecolor(BG_PANEL)
    ax.plot(test_dates, y_test, label="Actual",    color=TEXT_PRIMARY, linewidth=2)
    ax.plot(test_dates, pred,   label="Predicted", color=MODEL_COLORS[label],
            linewidth=1.8, linestyle="--")
    ax.axhline(0, color=TEXT_MUTED, linewidth=0.9, linestyle=":")
    da  = directional_accuracy(y_test, pred)
    mae = mean_absolute_error(y_test, pred)
    ax.set_title(f"{label}  |  DA={da:.2f}  MAE={mae:.4f}",
                 color=MODEL_COLORS[label], fontweight="bold", fontsize=10)
    ax.set_ylabel("Delta prob", color=TEXT_PRIMARY, fontsize=9)
    ax.legend(loc="upper right", facecolor=BG_PANEL, edgecolor=SPINE_COLOR,
              labelcolor=TEXT_PRIMARY, fontsize=8)
    for spine in ax.spines.values(): spine.set_edgecolor(SPINE_COLOR)
    ax.tick_params(colors=TEXT_MUTED)
    ax.grid(axis="y", color=GRID_COLOR, linewidth=0.7)
    ax.set_axisbelow(True)

axes[-1].set_xlabel("Date", color=TEXT_PRIMARY)
plt.xticks(rotation=25, ha="right")
fig.suptitle("Test-set predictions vs actuals\nDaily delta Polymarket Trump probability — Traditional-media features only",
             color=TEXT_PRIMARY, fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
metrics_plot = ["MAE", "RMSE", "Dir. Accuracy", "R2"]
model_names  = list(MODEL_COLORS.keys())
colors_list  = list(MODEL_COLORS.values())

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
fig.patch.set_facecolor(BG_DARK)

for ax, metric in zip(axes, metrics_plot):
    ax.set_facecolor(BG_PANEL)
    vals = [test_summary.loc[m, metric] for m in model_names]
    bars = ax.bar(range(len(model_names)), vals, color=colors_list, alpha=0.85, width=0.6)

    if metric == "Dir. Accuracy":
        ax.axhline(0.5, color=TEXT_MUTED, linestyle="--", linewidth=1.2, label="Coin flip (0.5)")
    if metric == "R2":
        ax.axhline(0, color=TEXT_MUTED, linestyle="--", linewidth=1.2, label="Naive / mean pred.")

    for bar, val in zip(bars, vals):
        offset = abs(val) * 0.03 + 0.001
        va     = "bottom" if val >= 0 else "top"
        y_pos  = bar.get_height() + offset if val >= 0 else bar.get_height() - offset
        ax.text(bar.get_x() + bar.get_width()/2, y_pos,
                f"{val:.3f}", ha="center", va=va, fontsize=7.5, color=TEXT_MUTED)

    ax.set_title(metric, color=TEXT_PRIMARY, fontweight="bold")
    ax.set_xticks(range(len(model_names)))
    ax.set_xticklabels([m.replace(" ", "\n") for m in model_names],
                       fontsize=7.5, color=TEXT_MUTED)
    for spine in ax.spines.values(): spine.set_edgecolor(SPINE_COLOR)
    ax.tick_params(colors=TEXT_MUTED)
    ax.grid(axis="y", color=GRID_COLOR, linewidth=0.7)
    ax.set_axisbelow(True)
    if metric in ("Dir. Accuracy", "R2"):
        ax.legend(facecolor=BG_PANEL, edgecolor=SPINE_COLOR, labelcolor=TEXT_PRIMARY, fontsize=8)

fig.suptitle("Test-set metric comparison — Traditional-media-only baseline",
             color=TEXT_PRIMARY, fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()